In [1]:
import sys
sys.path.insert(1, '../scripts/')
from preprocess import preprocess

First, format the project input files and generate the environment

In [2]:
# # downloaded project inputs
# input_data_path, build_files_path = preprocess.unpack_files(data_files = '/data2/hratch/human_me/data.zip', 
#                                         build_files = '/data2/hratch/human_me/build_files.zip', 
#                                         data_out = '/data2/hratch/human_me/raw')
# preprocess.create_environment(input_data_path, build_files_path, root_path = '/home/hratch/Projects/human_me/',
#                               processed_data_path = '/data2/hratch/human_me/processed/', 
#                               n_cores = 20)

In [3]:
from preprocess import correct_inputs 

full model

In [4]:
# correct_inputs.correct_model(model_file = '/data2/hratch/human_me/input_files/recon2_2.xml', 
#                  psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')

# correct_inputs.correct_psim(psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')
# # # optional - only if you want to express non-machinery proteins
# # correct_inputs.check_non_machinery(nonmachinery_file = '/data2/hratch/human_me/input_files/non_machinery.txt')

# from expression import build_me_model
# me_model, builder = build_me_model.build_me(minimal_proteome = False, compress_mrna = False)

# import pickle
# lp_path = '/data2/hratch/human_me/test_lp/'
# with open(lp_path + 'me_model.pickle', 'wb') as handle:
#     pickle.dump(me_model, handle)

toy model

In [5]:
# correct_inputs.correct_model(model_file = '/data2/hratch/human_me/input_files/toy_model.xml', 
#                  psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')

# correct_inputs.correct_psim(psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')

# from expression import build_me_model
# toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
#                                                 model_id = 'toy_me_model')

# import pickle
# lp_path = '/data2/hratch/human_me/test_lp/'
# with open(lp_path + 'toy_me_model.pickle', 'wb') as handle:
#     pickle.dump(toy_me_model, handle)



# Check

In [6]:
# from expression import build_me_model
# toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
#                                                 unmodeled_protein_frac = None,
#                                                 model_id = 'toy_me_model')
# jabba = True
# if jabba:
#     for r in toy_me_model.reactions:
#         if ('EX_' in r.id and r.compartments == {'b'} and r.bounds == (float('-inf'), float('inf'))):
#             r._lower_bound = -1000
#             r._upper_bound = 1000
            
# sln, stat, _ = toy_me_model.solve_lp(mu_val = 1e-9)

In [7]:
# import pandas as pd
# res = pd.DataFrame(data = {'reaction_fluxes': sln[:len(toy_me_model.reactions)]})
# res.index = [r.id for r in toy_me_model.reactions]

In [8]:
# res.loc[[i for i in res.index if 'biomass' in i],:]

In [9]:
# import pickle
# lp_path = '/data2/hratch/human_me/test_lp/'
# with open(lp_path + 'working_version_' + str(0) + '.pickle', 'wb') as handle:
#     pickle.dump(toy_me_model, handle)

# S = toy_me_model.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')
# fn = '/data2/hratch/human_me/test_lp/S_matrix.h5'
# S.to_hdf(fn, key = str(0), mode = 'w')

# Check dummy

In [12]:
from expression import build_me_model

dummy_model = build_me_model.build_me(non_machinery = [], minimal_proteome = True, 
                       compress_mrna = False, unmodeled_protein_frac = 0.05, model_id = 'HUMAN_ME_MODEL')[0]
toy_model = build_me_model.build_me(non_machinery = [], minimal_proteome = True, 
                       compress_mrna = False, unmodeled_protein_frac = None, model_id = 'HUMAN_ME_MODEL')[0]




In [13]:
res = {'dummy': {'model': dummy_model}, 'no_dummy': {'model': toy_model}}
for mt in res:
    sln, stat, _ = res[mt]['model'].solve_lp(mu_val = 1e-9)
    ir = res[mt]['model'].infeasible_reactions(1e-9, sln, stat)
    
    res[mt]['sln'] = sln
    res[mt]['stat'] = stat
    res[mt]['infeasible_reactions'] = ir
    
    


Getting MINOS parameters...
Done in 227.369 seconds with status 1
Getting MINOS parameters...
Done in 206.571 seconds with status 0


../scripts/core/model.py:242 UserWarning: There is a discrepancy between the solver status and reactions that violate bound constraints


In [63]:
import pandas as pd
res_df = pd.DataFrame(columns = res.keys(), 
                      index = sorted(set([r.id for r in res['dummy']['model'].reactions] + [r.id for r in res['no_dummy']['model'].reactions])))

same = sorted(set([r.id for r in res['dummy']['model'].reactions]).intersection([r.id for r in res['no_dummy']['model'].reactions]))


idx = res_df.index.copy()
for r_id in idx: 
    for k in res:
        if r_id in same:
            res_df.loc[r_id, k] = res[k]['sln'][res[k]['model'].reactions.index(r_id)]
        else:
            if r_id in [r.id for r in res[k]['model'].reactions]:
                if k == 'dummy':
                    if r_id[-2:] == '_F':
                        ids = [r_id, r_id.replace('_F', '_R')]
                        res_df.loc[r_id.replace('_F', ''), 'dummy'] = res_df.loc[ids[0],'no_dummy'] - res_df.loc[ids[1],'no_dummy']
                        res_df.drop(index = ids, inplace = True)
                    elif r_id[-2:] == '_R':
                        pass
                else:
                    res_df.loc[r_id, k] = res[k]['sln'][res[k]['model'].reactions.index(r_id)]
res_df['diff'] = abs(res_df['dummy'] - res_df['no_dummy'])
fail = res_df[(res_df['diff'] > 1e-20 )| (res_df['dummy'].isna())]

In [119]:
tol = max([abs(i) for i in res['no_dummy']['infeasible_reactions'].values()])

In [120]:
{k:v for k,v in res['dummy']['infeasible_reactions'].items() if abs(v) >= tol}

{'HGNC:10350_folded_protein_c_POLYUBIQUITINATIONc': -1.084529463373244e-18,
 'HGNC:10350_folded_protein_c_PROTEASOMAL_DEGRADATIONc': -1.084529463373244e-18,
 'HGNC:17094_folded_protein_c_POLYUBIQUITINATIONc': -1.084529463373244e-18,
 'HGNC:17094_folded_protein_c_PROTEASOMAL_DEGRADATIONc': -1.084529463373244e-18,
 'HGNC:10354_folded_protein_c_POLYUBIQUITINATIONc': -1.084529463373244e-18,
 'HGNC:10354_folded_protein_c_PROTEASOMAL_DEGRADATIONc': -1.084529463373244e-18,
 'HGNC:12458_processed_folded_protein_c_POLYUBIQUITINATIONc': -1.084529463373244e-18,
 'HGNC:12458_processed_folded_protein_c_PROTEASOMAL_DEGRADATIONc': -1.084529463373244e-18}

In [129]:
{k:v for k,v in res['dummy']['infeasible_reactions'].items() if math.isnan(v)}

{}

In [130]:
[i for i in res['dummy']['sln'] if math.isnan(i)]

[]

In [134]:
test = pd.Series(res['dummy']['sln'])

In [65]:
orphans = [r.id for r in res['no_dummy']['model'].reactions if len(r.genes) ==0]

print(len(set(fail[fail['dummy'].isna()].index).intersection(orphans)))
print(fail[fail['dummy'].isna()].shape[0])

287
297


In [76]:
len(set(orphans).intersection([r.id for r in res['no_dummy']['model'].boundary]))

87

In [81]:
all_fail = res_df.index

In [100]:
ndf = res['no_dummy']['infeasible_reactions'].keys()

dmf = res['dummy']['infeasible_reactions'].keys()
# dmf = sorted(set(dmf).difference(ndf))

In [105]:
fail.shape

(3047, 3)

In [108]:
res_df[res['dummy']['infeasible_reactions'].keys()]

KeyError: "None of [Index(['ACOATA_R', 'ADK1m_R_0', 'C226CPT1_0', 'C226CPT2', 'CATm',\n       'ECOAH1m_R_0', 'FACOAL226_F', 'FAOXC2051843m_0', 'GGH_10FTHF5GLUl',\n       'GGH_10FTHF6GLUl',\n       ...\n       'HGNC:10703_UNFOLDr', 'HGNC:2230_UNFOLDr', 'HGNC:17079_UNFOLDr',\n       'HGNC:7670_UNFOLDr', 'HGNC:25847_UNFOLDr', 'HGNC:11323_UNFOLDr',\n       'HGNC:2090_UNFOLDr', 'HGNC:4430_UNFOLDr', 'HGNC:2092_UNFOLDr',\n       'tRNA_biomass_to_biomass'],\n      dtype='object', length=1565)] are in the [columns]"

In [98]:
fail[fail['dummy'].isna()].shape

(297, 3)

In [104]:
set(fail.index).intersection(dmf)

{'ADNt_F_0',
 'HGNC:10073_folded_protein_n_PROTEASOMAL_DEGRADATIONn',
 'HGNC:10075_folded_protein_n_PROTEASOMAL_DEGRADATIONn',
 'HGNC:10350_TRANSLATION_ELONGATIONc',
 'HGNC:10350_folded_protein_c_POLYUBIQUITINATIONc',
 'HGNC:10350_folded_protein_c_PROTEASOMAL_DEGRADATIONc',
 'HGNC:10354_TRANSLATION_ELONGATIONc',
 'HGNC:10354_folded_protein_c_POLYUBIQUITINATIONc',
 'HGNC:10354_folded_protein_c_PROTEASOMAL_DEGRADATIONc',
 'HGNC:10445_folded_protein_c_PROTEASOMAL_DEGRADATIONc',
 'HGNC:10530_folded_protein_c_PROTEASOMAL_DEGRADATIONc',
 'HGNC:10538_folded_protein_n_PROTEASOMAL_DEGRADATIONn',
 'HGNC:10664_folded_protein_n_PROTEASOMAL_DEGRADATIONn',
 'HGNC:10765_folded_protein_n_PROTEASOMAL_DEGRADATIONn',
 'HGNC:10766_folded_protein_n_PROTEASOMAL_DEGRADATIONn',
 'HGNC:10767_folded_protein_n_PROTEASOMAL_DEGRADATIONn',
 'HGNC:10768_folded_protein_n_PROTEASOMAL_DEGRADATIONn',
 'HGNC:10769_folded_protein_n_PROTEASOMAL_DEGRADATIONn',
 'HGNC:10770_folded_protein_n_PROTEASOMAL_DEGRADATIONn',
 'HGNC:

In [ ]:
fail.intersect